# MoA Prediction — Fold and drug diagnostics

Companion of `moa-training.ipynb`. It runs in a few minutes, **without GPU and without retraining**, on the files that the training notebook already saved in its output folder. It answers four questions about a drug-aware CV run:

1. **Why is one fold so much worse than the others?** Fold sizes, losses per fold, and how much of the difference comes from very frequent drugs.
2. **Which drugs are the largest, in which fold do they fall, and which targets do they carry?**
3. **Which targets are dominated by a single drug,** and how badly are they predicted when that drug is held out?
4. **Which `HYBRID_MIN_ROWS` threshold gives balanced folds?** Simulated on the real labels in seconds, before spending GPU time on a new run.

## Inputs

The output folder of a run of `moa-training.ipynb` (attach it as a Dataset, or point `RUN_DIR` to it). Files used:
`oof_*.npy`, `oof_targets.npy`, `cv_fold_of_row.npy`, `train_drug_ids.npy`, `metadata.json` (all written by the training notebook, Steps 14.9 and 15.3).
The notebook only reads them; the only dependency besides `numpy` and `pandas` is `iterative-stratification` (needed by section 7).

## How to read the results

* Everything here is descriptive. A frequent drug being in the worst fold is a *candidate explanation*, not a proof: with 5 folds and a handful of very large drugs, fold composition and fold difficulty are confounded.
* Under `"grouped"` every drug is in a single fold. Under `"hybrid"` the rows of frequent drugs are spread over several folds; the "fold" of such a drug is then the most common one.

In [1]:
# Step 1.1 - install the multi-label stratification library (needed only by section 6; internet must be on)
!pip install -q iterative-stratification

In [2]:
# Step 1.2 - imports and configuration
import os
import glob
import json
import numpy as np
import pandas as pd
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# Folder with the outputs of the training run. None = look for it under /kaggle/input by file name.
RUN_DIR = None
FREQUENT_MIN_ROWS = 18        # a drug is "frequent" when it has MORE rows than this (same meaning as HYBRID_MIN_ROWS)
TOP_DRUGS = 12                # drugs listed in section 4
MIN_POS_DOMINANT = 50         # targets with at least this many positives are listed in section 5
THRESHOLDS = [6, 12, 18, 30, 60, 100]   # candidate HYBRID_MIN_ROWS values in section 6
N_FOLDS = 5
SEED = 42
CLIP_EPS = 1e-15


def find_input_directory(file_name):
    """The single folder under /kaggle/input that contains `file_name`."""
    matches = glob.glob(f"/kaggle/input/**/{file_name}", recursive=True)
    directories = sorted({os.path.dirname(path) for path in matches})
    if len(directories) != 1:
        raise FileNotFoundError(
            f"Expected exactly one folder containing '{file_name}' under /kaggle/input, "
            f"found {len(directories)}: {directories}. Attach the training-run output as an input, "
            "or set RUN_DIR explicitly.")
    return directories[0]


if RUN_DIR is None:
    RUN_DIR = find_input_directory("cv_fold_of_row.npy")
print(f"Run folder: {RUN_DIR}")

Run folder: /kaggle/input/datasets/marcomerlo37/moa-diagnostics-dataset


## 2. Load the run
Loads the saved OOF predictions, labels, fold of every row and drug of every row, and rebuilds the blend with the weights stored in `metadata.json`. Checks that all arrays are aligned.

In [3]:
# Step 2.1 - load
Y = np.load(f"{RUN_DIR}/oof_targets.npy")
fold_of_row = np.load(f"{RUN_DIR}/cv_fold_of_row.npy")
drug_ids = np.load(f"{RUN_DIR}/train_drug_ids.npy", allow_pickle=True)
with open(f"{RUN_DIR}/metadata.json") as metadata_file:
    metadata = json.load(metadata_file)
target_cols, weights = metadata["target_cols"], metadata["weights"]
oof = {name: np.load(f"{RUN_DIR}/oof_{name}.npy") for name in weights}

n_rows, n_targets = Y.shape
assert len(fold_of_row) == len(drug_ids) == n_rows, "row-level files are not aligned"
assert len(target_cols) == n_targets and all(P.shape == Y.shape for P in oof.values())
assert set(np.unique(fold_of_row)) == set(range(N_FOLDS)), "unexpected fold labels"
oof["blend"] = sum(weights[name] * oof[name] for name in weights)
print(f"CV scheme of the run: {metadata.get('cv_scheme', 'unknown')} | rows {n_rows} | targets {n_targets} | "
      f"drugs {len(np.unique(drug_ids))}")


# Step 2.2 - losses
def loss_matrix(Y, P, eps=CLIP_EPS):
    """Element-wise binary log loss, shape (rows, targets)."""
    P = np.clip(np.asarray(P, dtype=np.float64), eps, 1 - eps)
    return -(Y * np.log(P) + (1 - Y) * np.log(1 - P))


row_loss = {name: loss_matrix(Y, P).mean(axis=1) for name, P in oof.items()}
print("Pooled OOF log loss (rows-weighted): "
      + " | ".join(f"{name} {row_loss[name].mean():.5f}" for name in row_loss))

rows_per_drug = pd.Series(drug_ids).value_counts()
is_frequent = np.isin(drug_ids, rows_per_drug.index[rows_per_drug > FREQUENT_MIN_ROWS])
print(f"Frequent drugs (> {FREQUENT_MIN_ROWS} rows): {int((rows_per_drug > FREQUENT_MIN_ROWS).sum())} drugs, "
      f"{is_frequent.mean():.1%} of the rows")

CV scheme of the run: grouped | rows 21948 | targets 206 | drugs 3288
Pooled OOF log loss (rows-weighted): resnet 0.02006 | ft_transformer 0.02004 | catboost 0.02017 | blend 0.01951
Frequent drugs (> 18 rows): 9 drugs, 9.8% of the rows


## 3. Folds: size, composition and loss
One row per fold. `top5_drug_rows_share` is the share of the fold's rows that belong to its five largest drugs. A fold that is both larger and dominated by a few drugs is a candidate explanation for a loss outlier.

In [4]:
# Step 3.1 - per-fold table
rows = []
for fold in range(N_FOLDS):
    in_fold = fold_of_row == fold
    counts = pd.Series(drug_ids[in_fold]).value_counts()
    rows.append({"fold": fold, "rows": int(in_fold.sum()),
                 "size_vs_mean": in_fold.sum() / (n_rows / N_FOLDS) - 1,
                 "drugs": len(counts), "largest_drug": counts.index[0],
                 "largest_drug_rows": int(counts.iloc[0]),
                 "top5_drug_rows_share": counts.iloc[:5].sum() / in_fold.sum(),
                 **{f"loss_{name}": row_loss[name][in_fold].mean() for name in row_loss}})
per_fold = pd.DataFrame(rows)
print(per_fold.round(5).to_string(index=False))
worst_fold = int(per_fold["loss_blend"].idxmax())
others = per_fold.drop(index=worst_fold)
print(f"\nWorst fold (blend): {worst_fold} with {per_fold.loc[worst_fold, 'loss_blend']:.5f}, "
      f"{per_fold.loc[worst_fold, 'loss_blend'] / others['loss_blend'].mean() - 1:+.1%} vs the mean of the other folds")

 fold  rows  size_vs_mean  drugs largest_drug  largest_drug_rows  top5_drug_rows_share  loss_resnet  loss_ft_transformer  loss_catboost  loss_blend
    0  3987      -0.09172    658    83a9ea167                 18               0.01756      0.01929              0.01917        0.01951     0.01890
    1  4136      -0.05777    658    5628cb3ee                202               0.06310      0.01914              0.01879        0.01956     0.01874
    2  5512       0.25570    658    87d714366                718               0.28066      0.02274              0.02309        0.02210     0.02163
    3  3933      -0.10402    657    67c879e79                 19               0.01805      0.01966              0.01947        0.02005     0.01927
    4  4380      -0.00219    657    d08af5d4b                196               0.09566      0.01861              0.01870        0.01905     0.01834

Worst fold (blend): 2 with 0.02163, +15.0% vs the mean of the other folds


In [5]:
# Step 3.2 - is the outlier fold explained by the frequent drugs?
rows = []
for fold in range(N_FOLDS):
    in_fold = fold_of_row == fold
    freq, rest = in_fold & is_frequent, in_fold & ~is_frequent
    rows.append({"fold": fold,
                 "rows_frequent": int(freq.sum()), "share_frequent": freq.sum() / in_fold.sum(),
                 "loss_frequent": row_loss["blend"][freq].mean() if freq.any() else np.nan,
                 "rows_other": int(rest.sum()),
                 "loss_other": row_loss["blend"][rest].mean(),
                 "loss_all": row_loss["blend"][in_fold].mean()})
frequent_vs_other = pd.DataFrame(rows)
print(frequent_vs_other.round(5).to_string(index=False))

spread_all = frequent_vs_other["loss_all"].max() - frequent_vs_other["loss_all"].min()
spread_other = frequent_vs_other["loss_other"].max() - frequent_vs_other["loss_other"].min()
print(f"\nSpread of the fold loss (max - min): all rows {spread_all:.5f} | excluding frequent drugs {spread_other:.5f}")
if spread_other < 0.5 * spread_all:
    print("-> the spread shrinks a lot once the frequent drugs are excluded: they explain most of the outlier.")
else:
    print("-> the spread persists without the frequent drugs: the outlier is not (only) a frequent-drug effect.")

 fold  rows_frequent  share_frequent  loss_frequent  rows_other  loss_other  loss_all
    0              0         0.00000            NaN        3987     0.01890   0.01890
    1            202         0.04884        0.01537        3934     0.01891   0.01874
    2           1547         0.28066        0.02936        3965     0.01862   0.02163
    3             19         0.00483        0.03242        3914     0.01921   0.01927
    4            374         0.08539        0.01249        4006     0.01889   0.01834

Spread of the fold loss (max - min): all rows 0.00329 | excluding frequent drugs 0.00059
-> the spread shrinks a lot once the frequent drugs are excluded: they explain most of the outlier.


## 4. The largest drugs
For each of the largest drugs: rows, fold, share of that fold, loss of its rows, and the three targets with most positives among its rows. A large drug whose positives are concentrated in one or two targets makes those targets nearly unlearnable when the drug is held out.

In [6]:
# Step 4.1 - largest drugs
positives = pd.DataFrame(Y, columns=target_cols)
positives["drug"] = drug_ids
positives_per_drug = positives.groupby("drug").sum()

rows = []
for drug in rows_per_drug.index[:TOP_DRUGS]:
    in_drug = drug_ids == drug
    folds = fold_of_row[in_drug]
    main_fold = int(pd.Series(folds).mode().iloc[0])
    top_targets = positives_per_drug.loc[drug].nlargest(3)
    rows.append({"drug": drug, "rows": int(in_drug.sum()), "folds_spanned": len(np.unique(folds)),
                 "main_fold": main_fold,
                 "share_of_main_fold": (in_drug & (fold_of_row == main_fold)).sum() / (fold_of_row == main_fold).sum(),
                 "loss_blend": row_loss["blend"][in_drug].mean(),
                 "top_targets (positives)": ", ".join(f"{t} ({int(v)})" for t, v in top_targets.items() if v > 0)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

     drug  rows  folds_spanned  main_fold  share_of_main_fold  loss_blend                                          top_targets (positives)
87d714366   718              1          2              0.1303      0.0320                 nfkb_inhibitor (718), proteasome_inhibitor (718)
9f80f3f77   246              1          2              0.0446      0.0301                                            hmgcr_inhibitor (246)
8b87a7a83   203              1          2              0.0368      0.0085                                              cdk_inhibitor (203)
5628cb3ee   202              1          1              0.0488      0.0154                                             egfr_inhibitor (202)
d08af5d4b   196              1          4              0.0447      0.0082                                          tubulin_inhibitor (196)
292ab2c28   194              1          2              0.0352      0.0124                                                                 
d50f18348   186            

## 5. Targets dominated by a single drug
`top_drug_share` is the share of the target's positives that come from its single most frequent drug. `train_pos_when_top_drug_held_out` is how many positives of the target remain in the training part of the fold where that drug is validated. `expected/observed` is the sum of the blend's predicted probabilities divided by the observed positives (1 = calibrated, ≪ 1 = under-predicted).

A high `top_drug_share`, few remaining training positives and `expected/observed` far below 1 together describe a target that the drug-aware CV *cannot* learn in that fold. If the test set contains that drug, the real situation is different.

In [7]:
# Step 5.1 - dominated targets
n_pos = Y.sum(axis=0)
rows = []
for t, name in enumerate(target_cols):
    if n_pos[t] < MIN_POS_DOMINANT:
        continue
    per_drug = positives_per_drug[name]
    top_drug = per_drug.idxmax()
    top_fold = int(pd.Series(fold_of_row[drug_ids == top_drug]).mode().iloc[0])
    rows.append({"target": name, "n_pos": int(n_pos[t]), "n_drugs": int((per_drug > 0).sum()),
                 "top_drug": top_drug, "top_drug_share": per_drug.max() / n_pos[t],
                 "train_pos_when_top_drug_held_out": int(n_pos[t] - Y[fold_of_row == top_fold, t].sum()),
                 "expected/observed": oof["blend"][:, t].sum() / n_pos[t]})
dominated = pd.DataFrame(rows).sort_values("top_drug_share", ascending=False)
print(dominated.head(25).round(3).to_string(index=False))

heavy = dominated["top_drug_share"] >= 0.5
print(f"\nTargets with >= {MIN_POS_DOMINANT} positives: {len(dominated)}; "
      f"{int(heavy.sum())} have >= 50% of their positives from one drug")
if heavy.any() and (~heavy).any():
    print(f"Median expected/observed: dominated {dominated.loc[heavy, 'expected/observed'].median():.2f} | "
          f"others {dominated.loc[~heavy, 'expected/observed'].median():.2f}")

                                   target  n_pos  n_drugs  top_drug  top_drug_share  train_pos_when_top_drug_held_out  expected/observed
                     proteasome_inhibitor    726        4 87d714366           0.989                                 8              0.137
                          hmgcr_inhibitor    283        7 9f80f3f77           0.869                                31              0.254
                           nfkb_inhibitor    832       19 87d714366           0.863                                90              0.200
                            raf_inhibitor    223        9 d1b47f29d           0.798                                39              0.398
                            kit_inhibitor    273       17 d50f18348           0.681                                74              0.429
                           flt3_inhibitor    279       17 d50f18348           0.667                                81              0.481
                          pdgfr_inhibitor

## 6. Choosing `HYBRID_MIN_ROWS`
For each candidate threshold the hybrid split is simulated on the real labels (drugs with *more* rows than the threshold are split row by row, the others are kept whole), exactly as `make_split_groups` / `grouped_multilabel_splits` do in the training notebook (the two functions are copied here). The first line, "pure grouped", is the protocol of the run being analysed.

* `max_size_dev`: largest deviation of a fold's size from the mean (the training notebook warns above 15%).
* `val_rows_seen_drug`: average share of validation rows whose drug is also in the training fold. Row-wise CV is ~100%, pure grouped 0%. The lower the threshold, the closer the CV gets to the optimistic row-wise one.
* `targets_without_pos_in_val`: average number of targets with no positive in a validation fold.

Pick the **largest** threshold that brings `max_size_dev` to an acceptable level (a few %): it keeps as much of the drug-aware protocol as possible. This is a trade-off, not an optimum: neither extreme can be shown to be the faithful one.

In [8]:
# Step 6.1 - simulate hybrid splits for several thresholds (functions copied from moa-training.ipynb, Step 2.5)
def grouped_multilabel_splits(Y, groups, n_splits, seed):
    unique_groups, group_of_row = np.unique(groups, return_inverse=True)
    group_labels = np.zeros((len(unique_groups), Y.shape[1]), dtype=np.float64)
    np.maximum.at(group_labels, group_of_row, Y)
    splitter = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = []
    for _, val_groups in splitter.split(np.zeros((len(unique_groups), 1)), group_labels):
        is_val = np.isin(group_of_row, val_groups)
        train_idx, val_idx = np.where(~is_val)[0], np.where(is_val)[0]
        assert set(groups[train_idx]).isdisjoint(groups[val_idx]), "a group is split across folds"
        splits.append((train_idx, val_idx))
    return splits


def make_hybrid_groups(drug_ids, min_rows):
    _, drug_of_row, rows_per_drug = np.unique(drug_ids, return_inverse=True, return_counts=True)
    frequent = rows_per_drug[drug_of_row] > min_rows
    row_groups = np.char.add("row_", np.arange(len(drug_ids)).astype(str))
    return np.where(frequent, row_groups, drug_ids), frequent


rows = []
for min_rows in [None, *THRESHOLDS]:
    if min_rows is None:
        groups, frequent = drug_ids, np.zeros(n_rows, dtype=bool)
    else:
        groups, frequent = make_hybrid_groups(drug_ids, min_rows)
    splits = grouped_multilabel_splits(Y, groups, N_FOLDS, SEED)
    sizes = np.array([len(val_idx) for _, val_idx in splits])
    rows.append({
        "scheme": "pure grouped" if min_rows is None else f"hybrid, > {min_rows} rows",
        "rows_split_by_row": frequent.mean(),
        "fold_sizes": sizes.tolist(),
        "max_size_dev": np.abs(sizes / sizes.mean() - 1).max(),
        "val_rows_seen_drug": np.mean([np.isin(drug_ids[va], drug_ids[tr]).mean() for tr, va in splits]),
        "targets_without_pos_in_val": np.mean([(Y[va].sum(axis=0) == 0).sum() for _, va in splits]),
    })
threshold_table = pd.DataFrame(rows)
print(threshold_table.round(3).to_string(index=False))

            scheme  rows_split_by_row                     fold_sizes  max_size_dev  val_rows_seen_drug  targets_without_pos_in_val
      pure grouped              0.000 [3987, 4136, 5512, 3933, 4380]         0.256               0.000                        35.2
  hybrid, > 6 rows              0.220 [4380, 4349, 4539, 4388, 4292]         0.034               0.220                        29.8
 hybrid, > 12 rows              0.119 [4374, 4487, 4253, 4347, 4487]         0.031               0.119                        34.6
 hybrid, > 18 rows              0.098 [4386, 4356, 4371, 4383, 4452]         0.014               0.098                        35.0
 hybrid, > 30 rows              0.097 [4348, 4268, 4438, 4463, 4431]         0.028               0.097                        35.0
 hybrid, > 60 rows              0.097 [4348, 4268, 4438, 4463, 4431]         0.028               0.097                        35.0
hybrid, > 100 rows              0.097 [4348, 4268, 4438, 4463, 4431]         0.028 